# Spherical Harmonic Based HRTF Interpolation

Spherical harmonics are orthogonal spherical basis functions that have many use cases in acoustics signal processing. Most notably, the real-valued spherical harmonics are the basis of Ambisonics sound field capture and reproduction systems used in many immersive audio formats. But more generally, spherical harmonics are used to describe, analyze, interpolate, and extrapolate sound fields. Examples for this are estimating the direction of arrival, diffuseness, or directional decay time of sound fields in rooms.

In this notebook, we will look at another common use case: the representation, interpolation and rotation of so called head-related transfer functions (HRTFs). These transfer functions describe the sound propagation from a free field sound source to the left and right ear of a listener. HRTFs are fundamental for immersive audio rendering via headphones and are usually measured for sound sources distributed on a spherical sampling grid - which makes it natural to describe and process them in the spherical harmonic domain.

We will use the [spharpy](https://spharpy.readthedocs.io/) Python package for spherical harmonic processing and take HRTF processing as a chance to introduce core concepts of spharpy and [pyfar](https://pyfar.org). For a complete documentation of the related packages and more examples, please visit [pyfar.org](https://pyfar.org).

We recommend to visit the Notebooks about pyfar [audio](https://pyfar-gallery.readthedocs.io/en/latest/gallery/interactive/pyfar_audio_objects.html) and [coordinate](https://pyfar-gallery.readthedocs.io/en/latest/gallery/interactive/pyfar_coordinates.html) objects, and the Notebook about [binaural synthesis](https://pyfar-gallery.readthedocs.io/en/latest/gallery/static/binaural_synthesis.html) before continuing. We are also assuming that you are familiar with the basic concepts of spherical harmonics, which are for example detailed in [1] and [2].

[1] Rafaely, B. (2019). Fundamentals of spherical array processing (2nd ed.). Springer. https://doi.org/10.1007/978-3-319-99561-8

[2] Zotter, F., & Frank, M. (2019). Ambisonics. A practical 3D audio theory for recording, studio production, sound reinforcement, and virtual reality (Vol. 19). Springer Open. https://doi.org/10.1007/978-3-030-17207-7



In [ ]:
import spharpy
import pyfar as pf
import numpy as np
import matplotlib.pyplot as plt
import pooch
from pyfar.plot.ticker import MultipleFractionFormatter, MultipleFractionLocator
%matplotlib inline

## Load HRTFs

Lets first load an HRTF data set as a pyfar *Signal* and the source positions as a pyfar *Coordinates* object. The code below, downloads a publicly available HRTF dataset and loads it into the variables `hrirs` and `sources`.

In [ ]:
# Leave this as it is: This is the URL from which the data will be downloaded
# and a hash for checking if the download worked.
url = 'https://github.com/pyfar/files/raw/refs/heads/main/education/VAR_TUB/FABIAN_HRIR_measured_HATO_0.sofa?download='
hash = '83ebbcd9a09d17679b95d201c9775438c0bb1199d565c3fc7a25448a905cdc3c'

file = pooch.retrieve(
    url, hash, fname='FABIAN_HRIR_measured_HATO_0.sofa', path=None)

# load HRIRs and source positions
hrirs, sources, _ = pf.io.read_sofa(file)

## Inspect HRTFs and Source Positions

Lets quickly inspect the data, we are working with by looking at the channel shape (`cshape`) of the loaded objects. The Signal contains HRIRs for $Q=11950$ source positions and $2$ ears (the left ear data is contained in `hrirs[:, 0]`, and the right ear data in `hrirs[:, 1]`)

In [ ]:
hrirs.cshape

Accordingly, $11950$ source positions are stored in the Coordinates object

In [ ]:
sources.cshape

A quick plot of the source positions shows that they all have the same radius and represent a full-spherical sampling grid

In [ ]:
spharpy.plot.scatter(sources)
plt.show()

Accordingly, we can store them in a spharpy [SamplingSphere](https://spharpy.readthedocs.io/en/stable/classes/spharpy.coordinates.html) object which is specifically intended for point distributions on a sphere.

In [ ]:
sources = spharpy.SamplingSphere.from_coordinates(sources)

## Background

We interpret it as a function on the sphere discretely sampled at the source positions. This yields the following vector 

$$\mathbf{h} = [h_1, h_2, ..., h_Q]^\mathrm{T}$$

that represents a single sample of the HRIR or a single frequency bin of the HRTF for one ear and **all** $Q$ source positions.

We can apply the spherical harmonic transform - also referred to as spherical Fourier transform - to $\mathbf{h}$ to get the vector of $(N+1)^2$ spherical harmonic coefficients

$$\mathrm{h}_{nm} = \mathbf{Y}^\dagger \, \mathbf{h} = [h_{0,0}, h_{1,-1}, h_{1,-1}, h_{1,-1}, ..., , h_{N,N}]^\mathrm{T}$$

where order $n$ and degree $m$ up to the maximum spherical harmonic order $N$.

In the above $\mathbf{Y} \isin \mathbb{R}^{Q \times (N+1)^2}$ denotes the matrix

$$
\mathbf{Y} = \begin{bmatrix}
Y_0^0(\theta_1, \phi_1) & Y_1^{-1}(\theta_1, \phi_1) & Y_1^0(\theta_1, \phi_1)  & \cdots & Y_N^N(\theta_1, \phi_1) \\
Y_0^0(\theta_2, \phi_2) & Y_1^{-1}(\theta_2, \phi_2) & Y_1^0(\theta_2, \phi_2)  & \cdots & Y_N^N(\ \theta_2, \phi_2) \\
\vdots & \vdots & \vdots & \ddots & \vdots \\
Y_0^0(\theta_Q, \phi_Q) & Y_1^{-1}(\theta_Q, \phi_Q) & Y_1^0(\theta_Q, \phi_Q)  & \cdots & Y_N^N(\theta_Q, \phi_Q) \end{bmatrix}
$$

containing the real valued spherical harmonic basis functions $Y_n^m$ evaluated at the colatitude $\theta$ and the azimuth $\phi$ angles of the source positions and $(\cdot)^\dagger$ is the pseudo inverse.

Once $\mathbf{h}_{nm}$ is computed, it can be used to interpolate the HRTFs at any source position $\hat{\theta}$, $\hat{\phi}$ by means of the inverse spherical harmonic transform

$$\hat{\mathbf{h}} = \mathbf{\hat{Y}} \, \mathbf{h}_{nm}$$

For a single target source position $\mathbf{\hat{Y}}$ is

$$\mathbf{\hat{Y}} = [Y_0^0(\hat{\theta},\hat{\phi}), Y_1^{-1}(\hat{\theta},\hat{\phi}), Y_1^0(\hat{\theta},\hat{\phi}), Y_1^1(\hat{\theta},\hat{\phi}), ..., Y_N^N(\hat{\theta},\hat{\phi})]$$

but it can contain any number of source positions in general.

Because $\mathbf{h}$ contains data for a single sample or frequency bin, the above is done separately for *each* sample or frequency bin.

## Spherical Harmonic Definition

Before computing $\mathrm{Y}$ and its pseudo inverse, we need to decide on the maximum spherical harmonic order and the spherical harmonic definition.

This specific sampling grid supports spherical harmonic processing up to an order of approximately $N=32$ and for simplicity, we use the default spherical harmonic definition of spharpy.

The below creates a [SphericalHarmonicDefinition](https://spharpy.readthedocs.io/en/stable/theory/spherical_harmonic_definition.html) object that conveniently documents, how the used spherical harmonics are defined.

In [ ]:
# specify sh order and definition
n_max = 16
sh_definition = spharpy.SphericalHarmonicDefinition(n_max)

print(f'{sh_definition.basis_type = }')
print(f'{sh_definition.normalization = }')
print(f'{sh_definition.condon_shortley = }')
print(f'{sh_definition.channel_convention = }')
print(f'{sh_definition.n_max = }')

## Spherical Harmonic Matrix

We can now create a `SphericalHarmonics` object.

In [ ]:
sh = spharpy.SphericalHarmonics.from_definition(
    sh_definition, sources, inverse_method="pseudo_inverse")

Among other properties, it provides the spherical harmonic basis matrix $\mathbf{Y}$

In [ ]:
sh.basis.shape

and its pseudo inverse $\mathbf{Y}^\dagger$

In [ ]:
sh.basis_inv.shape

These matrices are computed on demand and stored inside the `SphericalHarmonics` objects. They can be reused across multiple operations and are only recomputed if any property of the SphericalHarmonic object changes, for example if you set a different maximum spherical harmonic order.

## Spherical Harmonic Transform

We can now compute $\mathrm{h}_{nm} = \mathbf{Y}^\dagger \, \mathbf{h}$. We do this with a single [pyfar.matrix_multiplication](https://pyfar.readthedocs.io/en/stable/classes/pyfar.audio.html#pyfar.matrix_multiplication) that performs the transform for all samples and both ears.

In this case, we perform the transform in the time domain. Because the Fourier transform and spherical harmonic transform are linear operations, we could as well perform the transform in the frequency domain.

In [ ]:
hrirs_nm = pf.matrix_multiplication((sh.basis_inv, hrirs), domain='time').T
print(type(hrirs_nm))
print(f'{hrirs_nm.cshape = }')
print(f'{hrirs_nm.n_samples = }')

Alternatively, the matrix multiplication operator `@` can be used to perform the transform. Note that using the operator always performs the multiplication in the frequency domain. 

In [ ]:
hrirs_nm = (sh.basis_inv @ hrirs).T

The above yields a pyfar Signal object. It has two channels (left and right ear), $(N+1)^2$ (`(n_max+1)**2`) spherical harmonic coefficients, and 256 samples. Lets convert it to a `SphericalHarmonicSignal`, which conveniently stores the spherical harmonic data and sampling rate along with the spherical harmonic definition.

In [ ]:
hrirs_nm = spharpy.SphericalHarmonicSignal.from_definition(
    sh_definition, hrirs_nm.time, hrirs_nm.sampling_rate)

print(f'{hrirs_nm.cshape = }')
print(f'{hrirs_nm.n_max = }')

For illustration, lets plot the left and right ear spherical harmonic signal of order 1 and degree 1. This is contained in `hrirs_nm[:, 1]` and contains the 'left/right'-component of the HRTF data set.

In [ ]:
ax = pf.plot.time_freq(hrirs_nm[:, 1], label=['left ear', 'right ear'])
ax[1].legend()
plt.show()

More specifically, the visualized data corresponds to the first order dipole moment oriented in the $y$-axis as visualized below.

In [ ]:
axs, _, cb = spharpy.plot.balloon_wireframe(sources, sh.basis[:, 1])
cb.ax.yaxis.set_major_locator(MultipleFractionLocator(1, 2, base=np.pi))
cb.ax.yaxis.set_major_formatter(MultipleFractionFormatter(1, 2, base=np.pi, base_str=r'\pi'))

## Spherical Harmonic Rotation

A common manipulation is a rotation of the data in the spherical harmonic domain. One use case in Ambisonics is to rotate the sound field to counter head rotations of the listener during headphone playback. This creates a naturally stable sound scene that the listener can explore with head rotations.

Mathematically, this is realized by a multiplication with a rotation matrix $\mathbf{R}$

$$\mathbf{h}_{nm,\mathrm{rot}} = \mathbf{R} \, \mathbf{h}_{nm}$$

In spharpy, arbitrary rotations can be applied with the [SphericalHarmonicRotation](https://spharpy.readthedocs.io/en/stable/modules/spharpy.transforms.html#spharpy.transforms.SphericalHarmonicRotation) class. Lets create a rotation by 90 degrees about the z-axis.

In [ ]:
rotation = spharpy.transforms.SphericalHarmonicRotation.from_euler(
    'z', [np.pi / 2])

One way to apply the rotation would be to generate the rotation matrix $\mathbf{R}$ using

`R = rotation.as_spherical_harmonic_matrix(sh_definition)`

and perform the matrix multiplication introduced above.

In this example we will directly use the SphericalHarmonicRotation object and the SphericalHarmonicSignal

In [ ]:
hrirs_nm_rotated = rotation.apply(hrirs_nm)

## Inverse Spherical Harmonic Transform

We now perform the inverse transform introduced above as

$$\hat{\mathbf{h}} = \mathbf{Y} \, \mathbf{h}_{nm}$$

Note two things:

1. If $N$ is sufficently large, we get $\hat{\mathbf{h}} = \mathbf{h}$ after applying the inverse transform. HRTFs require $N>32$, which means that you will see differences between $\hat{\mathbf{h}}$ and $\mathbf{h}$ 
2. In the example below $\mathbf{Y}$ contains the source positions of the original HRTF data set. In general, it can contain one or multiple arbitrary source positions.

Again, the transform for all time samples can be done with a single call of [pyfar.matrix_multiplication](https://pyfar.readthedocs.io/en/stable/classes/pyfar.audio.html#pyfar.matrix_multiplication).

In [ ]:
print(sh.basis.shape)
print(hrirs_nm.time.shape)

In [ ]:
# obtain HRIRs using the inverse SH transform
hrirs_sh = pf.matrix_multiplication(
    (sh.basis, hrirs_nm), domain='time', axes=[(0, 1), (1, 0), (0, 1)])
hrirs_sh = pf.Signal(hrirs_sh, hrirs.sampling_rate)

# obtain rotated HRIRs using the inverse SH transform
hrirs_sh_rotated = pf.matrix_multiplication(
    (sh.basis, hrirs_nm_rotated), domain='time', axes=[(0, 1), (1, 0), (0, 1)])
hrirs_sh_rotated = pf.Signal(hrirs_sh_rotated, hrirs.sampling_rate)

hrirs_sh.cshape

## Plot the Results

### Single HRTF

Lets first plot a single HRTF before and after the spherical harmonic processing.

In [ ]:
# source position for plotting
find = pf.Coordinates.from_spherical_colatitude(
    90 / 180 * np.pi,  # azimuth angle
    90 / 180 * np.pi,  # colatitude angle
    1.7)               # radius
idx, _ = sources.find_nearest(find)
idx = idx[0]

# plot
ax = pf.plot.freq(
    hrirs[idx], label=['original: left', 'original:right'])
pf.plot.freq(
    hrirs_sh[idx], ls="--", label=['processed: left', 'processed: right'])

ax.set_ylim(-25, 25)
ax.legend(loc='lower left')
plt.show()

As mentioned above, you can see that the spherical harmonic processing introduced errors to the HRTF and the processed HRTFs will look different for each spherical harmonic order.

### Plot single Frequency

In [ ]:
frequency = 1000
idx = hrirs_sh.find_nearest_frequency(frequency)

pf.plot.use()
fig, axes = plt.subplots(2,1, subplot_kw={'projection': 'mollweide'})

axes[0].set_title('Left ear processed HRTF (no rotation)')
spharpy.plot.pcolor_map(
    sources, 20 * np.log10(np.abs(hrirs_sh.freq[:, 0, idx])),
    ax=axes[0])
axes[0].grid(True)

axes[1].set_title('Left ear processed HRTF (45 deg. rotation)')
spharpy.plot.pcolor_map(
    sources, 20 * np.log10(np.abs(hrirs_sh_rotated.freq[:, 0, idx])),
    ax=axes[1])
axes[1].grid(True)

plt.show()

## Exploration

Now that you have coded all the above, it's time to play. For example, you can try to the processing with different spherical harmonic orders to see how the order affects the results. You could also render the HRTFs and listen to the result via headphones. An example for this is given in the [binaural synthesis](https://pyfar-gallery.readthedocs.io/en/latest/gallery/static/binaural_synthesis.html) notebook.

# License notice

This notebook is licensed under CC BY 4.0

# Watermark

The following watermark might help others to install specific package versions that might be required to run the notebook. Please give at least the versions of Python, IPython, numpy , and scipy, major third party packagers (e.g., pytorch), and all used pyfar packages.

In [ ]:
%load_ext watermark
%watermark -v -m -p numpy,scipy,pyfar,sofar,nbgrader,watermark